## Impact of query

In [ ]:
import pandas as pd
import numpy as np

final_answers = pd.read_csv("Data/final_answer_df.csv")
queries = pd.read_csv("Data/query_formulation_df.csv")

In [ ]:
final_answers.head()

In [ ]:
queries.head()

In [ ]:
queries.columns

In [ ]:
## query characteristics
num_queries = []
num_unique_queries = []

query_lex_density = []
query_aoa = []
query_gulpease_index = []
query_num_typos = []
query_lev_dist = []

query_len = []
query_num_concepts = []
query_conc = []
query_stop_words_ratio = []

query_p10 = []

for _, row in final_answers.iterrows():
    uid = row["uid"]
    topic_number = row["topic_number"]
    session_queries = queries.loc[(queries["uid"]==uid) & (queries["topic_number"]==topic_number)]
    num_queries.append(len(session_queries["query"]))
    num_unique_queries.append(len(set(session_queries["query"])))
    query_lex_density.append(session_queries["lex_density"].dropna().mean())
    query_aoa.append(session_queries["mean_aoa"].dropna().mean())
    query_gulpease_index.append(session_queries["gulpease_index"].dropna().mean())
    query_num_typos.append(session_queries["num_typos"].dropna().mean())
    query_lev_dist.append(session_queries["levenshtein_dist"].dropna().mean())
    query_len.append(session_queries["query_len"].dropna().mean())
    query_num_concepts.append(session_queries["num_concepts"].dropna().mean())
    query_conc.append(session_queries["mean_concepts_conc"].dropna().mean())
    query_stop_words_ratio.append(session_queries["stop_word_ratio"].dropna().mean())
    query_p10.append(session_queries["precision@10"].dropna().mean())

final_answers["num_queries"] = num_queries
final_answers["num_unique_queries"] = num_unique_queries

final_answers["query_lex_density"] = query_lex_density
final_answers["query_aoa"] = query_aoa
final_answers["query_gulpease_index"] = query_gulpease_index
final_answers["query_num_typos"] = query_num_typos
final_answers["query_lev_dist"] = query_lev_dist

final_answers["query_len"] = query_len
final_answers["query_num_concepts"] = query_num_concepts
final_answers["query_conc"] = query_conc
final_answers["query_stop_words_ratio"] = query_stop_words_ratio

final_answers["query_p10"] = query_p10

In [ ]:
final_answers.to_csv("Data/final_answer_queries.csv", index=False, encoding="utf8")
final_answers.head()

## Impact of SERP navigation

In [ ]:
import pandas as pd

final_answers = pd.read_csv("Data/final_answer_df.csv")
serp = pd.read_csv("Data/results_navigation.csv")
navigation_quality = pd.read_csv("Data/navigation_quality.csv")
results_navigation = pd.read_csv("Data/results_navigation.csv")

viewed_results = pd.read_csv("Data/viewed_results_df.csv")
viewed_results["timestamp"] = pd.to_datetime(viewed_results["timestamp"])

explored_pages = pd.read_csv("Data/explored_pages.csv")
explored_pages["click_time"] = pd.to_datetime(explored_pages["click_time"])

partial_answers = pd.read_csv("Data/information_distillation_df.csv")
corpus = pd.read_excel("Data/analysed_corpus.xlsx")
relevance = pd.read_csv("Data/relevance.csv")

In [ ]:
navigation_quality.columns

In [ ]:
results_navigation.columns

In [ ]:
viewed_results.head(5)

In [ ]:
explored_pages.head(5)

In [ ]:
## SERP characteristics

SERP_dwell_time = []
snippet_dwell_time = []
click_depth = []
inspect_depth = []

view_depth = []
view_repeat = []
unique_view_count = []

viewed_aoa = []
viewed_len = []
viewed_gulpease_index = []
viewed_conc = []

for _, row in final_answers.iterrows():
    uid = row["uid"]
    topic_number = row["topic_number"]
    session_serp_nav = navigation_quality.loc[(navigation_quality["uid"]==uid) & (navigation_quality["topic_number"]==topic_number)]
    session_serps = results_navigation.loc[(results_navigation["uid"]==uid) & (results_navigation["topic_number"]==topic_number)]
    session_views = viewed_results.loc[(viewed_results["uid"]==uid) & (viewed_results["topic_number"]==topic_number)]

    viewed_snippets = session_serps.loc[session_serps["view_count"]>0]
    unique_views = list(viewed_snippets["url"].unique())

    serp_dwell_times = []
    serp_snippet_dwell_time = []
    for serp_id in session_views["serp_id"].unique():
        serp_views = session_views.loc[session_views["serp_id"]==serp_id]

        serp_snippet_dwell_time.append(serp_views.loc[serp_views["viewType"]=="resultExposureEnded"]["viewDurationInSeconds"].sum()/len(serp_views["url"].unique()))

        serp_explored_pages = explored_pages.loc[explored_pages["serp_id"]==serp_id]
        if serp_explored_pages["click_time"].max() > serp_views["timestamp"].max():
            serp_dwell_time = ((serp_views["timestamp"].max() - serp_views["timestamp"].min()).total_seconds()) - serp_explored_pages[:-1]["viewDurationInSeconds"].sum()
        else:
            serp_dwell_time = ((serp_views["timestamp"].max() - serp_views["timestamp"].min()).total_seconds()) - serp_explored_pages["viewDurationInSeconds"].sum()
            
        if serp_dwell_time < 0:
            print(serp_id)
            raise ValueError("The total time spent on the SERP is less than the time spent engaging with the resources clicked on the SERP")
        elif serp_dwell_time == 0:
            print(serp_id)
            raise ValueError("The total time spent on the SERP is equal to the time spent engaging with the resources clicked on the SERP")
        else:
            serp_dwell_times.append(serp_dwell_time)

    unique_view_snippets = viewed_snippets.drop_duplicates(subset=["url"])
    viewed_aoa.append(unique_view_snippets["SERP_result_mean_aoa"].mean())
    viewed_len.append(unique_view_snippets["SERP_result_content_len"].mean())
    viewed_gulpease_index.append(unique_view_snippets["SERP_result_gulpease_index"].mean())
    viewed_conc.append(unique_view_snippets["SERP_result_mean_conc"].mean())
    
    SERP_dwell_time.append(sum(serp_dwell_times)/len(session_views["serp_id"].unique()))  
    snippet_dwell_time.append(sum(serp_snippet_dwell_time)/len(session_views["serp_id"].unique())) 
    view_depth.append(session_serp_nav["inspect_depth"].mean())
    click_depth.append(session_serp_nav["click_depth"].mean())
    unique_view_count.append(len(unique_views))
    view_repeat.append((viewed_snippets["view_count"].sum() - len(unique_views))/len(unique_views))

final_answers["SERP_dwell_time"] = SERP_dwell_time
final_answers["snippet_dwell_time"] = snippet_dwell_time
final_answers["click_depth"] = click_depth
final_answers["snippet_aoa"] = viewed_aoa
final_answers["snippet_len"] = viewed_len
final_answers["snippet_gulpease_index"] = viewed_gulpease_index
final_answers["snippet_conc"] = viewed_conc
final_answers["inspect_depth"] = view_depth
final_answers["snippet_repeat"] = view_repeat
final_answers["unique_snippet_count"] = unique_view_count

In [ ]:
final_answers.to_csv("Data/final_answer_serp.csv", index=False, encoding="utf8")
final_answers.head()

## Impact of page engagement

In [ ]:
import pandas as pd

final_answers = pd.read_csv("Data/final_answer_df.csv")
serp = pd.read_csv("Data/results_navigation.csv")
navigation_quality = pd.read_csv("Data/navigation_quality.csv")

explored_pages = pd.read_csv("Data/explored_pages.csv")
partial_answers = pd.read_csv("Data/information_distillation_df.csv")
corpus = pd.read_excel("Data/analysed_corpus.xlsx")
relevance = pd.read_csv("Data/relevance.csv")

In [ ]:
explored_pages.head()

In [ ]:
serp.columns

In [ ]:
corpus.columns

In [ ]:
relevance.head()

In [ ]:
navigation_quality.columns

In [ ]:
## Resource characteristics

explored_pages = explored_pages.loc[explored_pages["exploreType"]=="webpageClosed"]

num_unique_explored_pages = []
page_repeats = []
click_accuracy = []
click_recall = []

page_dwell_time = []
rel_page_dwell_time = []
irrel_page_dwell_time = []
ratio_rel_total_page_dwell_time = []

page_aoa = []
page_len = []
page_conc = []
page_gulpease_index = []

rel_page_aoa = []
rel_page_len = []
rel_page_conc = []
rel_page_gulpease_index = []

for _, row in final_answers.iterrows():
    uid = row["uid"]
    topic_number = row["topic_number"]
    session_explored_pages = explored_pages.loc[(explored_pages["uid"]==uid) & (explored_pages["topic_number"]==topic_number)]
    session_serp_navs = navigation_quality.loc[(navigation_quality["uid"]==uid) & (navigation_quality["topic_number"]==topic_number)]
    # print(uid, topic_number, len(session_explored_pages))
    corpus_pages = corpus.loc[corpus["url"].isin(set(session_explored_pages["url"]))]

    num_explored_pages = len(session_explored_pages)

    unique_pages = set(session_explored_pages["url"])
    num_unique_explored_pages.append(len(unique_pages))


    page_aoa.append(corpus_pages["mean_aoa"].dropna().mean())
    page_len.append(corpus_pages["content_len"].dropna().mean())
    page_conc.append(corpus_pages["mean_conc"].dropna().mean())
    page_gulpease_index.append(corpus_pages["gulpease_index"].dropna().mean())

    rel_pages = []
    irrel_pages = []
    page_dwell = []
    rel_page_dwell = []
    irrel_page_dwell = []

    for _, page in session_explored_pages.iterrows():
        url = page["url"]
        if page["relevance_score"] == 1:
            rel_page_dwell.append(page["viewDurationInSeconds"])
            rel_pages.append(url)
        elif page["relevance_score"] == 0:
            irrel_page_dwell.append(page["viewDurationInSeconds"])
            irrel_pages.append(url)
        else:
            print("Page has no relevance score??", page)
        page_dwell.append(page["viewDurationInSeconds"])

    if len(unique_pages) == 0:
        page_dwell_time.append(None)
        rel_page_dwell_time.append(None)
        irrel_page_dwell_time.append(None)
        ratio_rel_total_page_dwell_time.append(None)
        page_repeats.append(None)
    else:
        page_repeats.append((num_explored_pages - len(unique_pages))/len(unique_pages))
        page_dwell_time.append(np.mean(page_dwell))
        rel_page_dwell_time.append(np.mean(rel_page_dwell))
        irrel_page_dwell_time.append(np.mean(irrel_page_dwell))
        ratio_rel_total_page_dwell_time.append(sum(rel_page_dwell)/sum(page_dwell))

    click_accuracy.append(session_serp_navs["click_accuracy"].dropna().mean())
    click_recall.append(len(set(rel_pages))/len(relevance.loc[relevance["topic_number"]==topic_number]["doc"].unique()))

    corpus_rel_pages = corpus_pages.loc[corpus_pages["url"].isin(set(rel_pages))]
    rel_page_aoa.append(corpus_rel_pages["mean_aoa"].dropna().mean())
    rel_page_len.append(corpus_rel_pages["content_len"].dropna().mean())
    rel_page_conc.append(corpus_rel_pages["mean_conc"].dropna().mean())
    rel_page_gulpease_index.append(corpus_rel_pages["gulpease_index"].dropna().mean())

final_answers["page_repeats"] = page_repeats
final_answers["num_unique_explored_pages"] = num_unique_explored_pages
final_answers["click_accuracy"] = click_accuracy
final_answers["click_recall"] = click_recall

final_answers["page_dwell_time"] = page_dwell_time
final_answers["rel_page_dwell_time"] = rel_page_dwell_time
final_answers["irrel_page_dwell_time"] = irrel_page_dwell_time
final_answers["ratio_rel_total_page_dwell_time"] = ratio_rel_total_page_dwell_time

final_answers["page_aoa"] = page_aoa
final_answers["page_len"] = page_len
final_answers["page_conc"] = page_conc
final_answers["page_gulpease_index"] = page_gulpease_index

final_answers["rel_page_aoa"] = rel_page_aoa
final_answers["rel_page_len"] = rel_page_len
final_answers["rel_page_conc"] = rel_page_conc
final_answers["rel_page_gulpease_index"] = rel_page_gulpease_index

In [ ]:
final_answers.to_csv("Data/final_answer_pages.csv", index=False, encoding="utf8")
final_answers.head()

## Visualisation

### Queries

In [ ]:
import pandas as pd

final_answers_queries = pd.read_csv("Data/final_answer_queries.csv")

In [ ]:
from scipy.stats import spearmanr

categories = final_answers_queries.drop(columns=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "TaskDuration", "matched_keywords", "num_answer_matches", "answer_match_ratio"])
for category in categories:
    res = spearmanr(final_answers_queries["answer_match_ratio"], final_answers_queries[category], nan_policy="omit")
    print(f"Category: {category}; \tcoefficient: {res.statistic}; \tpval: {res.pvalue}")

In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

corr = final_answers_queries.drop(columns=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "TaskDuration", "matched_keywords", "num_answer_matches"], inplace=False).corr(method="spearman", numeric_only=True)

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(11, 9))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, cmap=cmap,
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, annot=True, fmt=".2f")


### SERP

In [ ]:
import pandas as pd

final_answers_serp = pd.read_csv("Data/final_answer_serp.csv")

In [ ]:
from scipy.stats import spearmanr

categories = final_answers_serp.drop(columns=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "TaskDuration", "matched_keywords", "num_answer_matches", "answer_match_ratio"])
for category in categories:
    res = spearmanr(final_answers_serp["answer_match_ratio"], final_answers_serp[category], nan_policy="omit")
    print(f"Category: {category}; \tcoefficient: {res.statistic}; \tpval: {res.pvalue}")

In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

corr = final_answers_serp.drop(columns=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "TaskDuration", "matched_keywords", "num_answer_matches"], inplace=False).corr(method="spearman", numeric_only=True)

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(11, 9))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, cmap=cmap,
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, annot=True, fmt=".2f")


### Pages

In [ ]:
import pandas as pd

final_answers_pages = pd.read_csv("Data/final_answer_pages.csv")

In [ ]:
from scipy.stats import spearmanr

categories = final_answers_pages.drop(columns=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "TaskDuration", "matched_keywords", "num_answer_matches", "answer_match_ratio"])
for category in categories:
    res = spearmanr(final_answers_pages["answer_match_ratio"], final_answers_pages[category], nan_policy="omit")
    print(f"Category: {category}; \tcoefficient: {res.statistic}; \tpval: {res.pvalue}")

In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

corr = final_answers_pages.drop(columns=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "TaskDuration", "matched_keywords", "num_answer_matches"], inplace=False).corr(method="spearman", numeric_only=True)

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(11, 9))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, cmap=cmap,
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, annot=True, fmt=".2f")


### All together

In [ ]:
import pandas as pd

final_answers_queries = pd.read_csv("Data/final_answer_queries.csv")
final_answers_serp = pd.read_csv("Data/final_answer_serp.csv")
final_answers_pages = pd.read_csv("Data/final_answer_pages.csv")

final_answers_combined = pd.merge(final_answers_queries, final_answers_serp, on=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "TaskDuration", "matched_keywords", "num_answer_matches", "answer_match_ratio"])
final_answers_combined = pd.merge(final_answers_combined, final_answers_pages, on=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "TaskDuration", "matched_keywords", "num_answer_matches", "answer_match_ratio"])
final_answers_combined.to_csv("Results/final_answers_combined.csv", index=False, encoding="utf8")
final_answers_combined.head()

In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

corr = final_answers_combined.drop(columns=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "matched_keywords", "num_answer_matches"], inplace=False).corr(method="spearman", numeric_only=True)

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(22, 18))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, cmap=cmap,
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, annot=True, fmt=".2f")


### Radar plots

In [ ]:
from driver import get_radar_plot
import pandas as pd

final_answers_queries = pd.read_csv("Data/final_answer_queries.csv")
final_answers_serp = pd.read_csv("Data/final_answer_serp.csv")
final_answers_serp = final_answers_serp.drop(columns = ["snippet_dwell_time"])

final_answers_pages = pd.read_csv("Data/final_answer_pages.csv")

final_answers_serp["click_accuracy"] = [final_answers_pages.loc[(final_answers_pages["uid"]==row["uid"]) & (final_answers_pages["topic_number"]==row["topic_number"])]["click_accuracy"].values[0] for _, row in final_answers_serp.iterrows()]
final_answers_pages = final_answers_pages.drop(columns = ["rel_page_gulpease_index", "rel_page_conc", "rel_page_len", "rel_page_aoa", "ratio_rel_total_page_dwell_time", "click_recall", "click_accuracy"])

final_answers_combined = pd.merge(final_answers_queries, final_answers_pages, on=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "TaskDuration", "matched_keywords", "num_answer_matches", "answer_match_ratio"])

for df_name, search_stage in zip([
    final_answers_queries, 
    final_answers_serp,
    final_answers_pages,
    ], ["queries", "nav", "distil"]):

    final_answers_trial = df_name

    user_category = []

    for _, row in final_answers_trial.iterrows():
        if row["answer_match_ratio"] == 1:
            user_category.append(1)
        else:
            user_category.append(0)

    final_answers_trial["user_category"] = user_category

    categories = final_answers_trial.drop(columns=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "matched_keywords", "num_answer_matches", "answer_match_ratio", "user_category", "TaskDuration"])
    get_radar_plot(df=final_answers_trial, search_stage=search_stage, categories=list(categories.columns), artefact="session", group = "user_category")

In [ ]:
import pandas as pd

final_answers_queries = pd.read_csv("Data/final_answer_queries.csv")
final_answers_serp = pd.read_csv("Data/final_answer_serp.csv")
final_answers_pages = pd.read_csv("Data/final_answer_pages.csv")

final_answers_combined = pd.merge(final_answers_queries, final_answers_serp, on=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "TaskDuration", "matched_keywords", "num_answer_matches", "answer_match_ratio"])
final_answers_combined = pd.merge(final_answers_combined, final_answers_pages, on = ["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "TaskDuration", "matched_keywords", "num_answer_matches", "answer_match_ratio"])
final_answers_combined.to_csv("Data/final_answer_combined.csv", index=False, encoding="utf8")

In [ ]:
final_answers_combined.head()

In [ ]:
import pandas as pd

final_answers_combined = pd.read_csv("Data/final_answer_combined.csv")
query_formulation = pd.read_csv("Data/query_formulation_df.csv")
navigation_quality = pd.read_csv("Data/navigation_quality.csv")
information_distillation = pd.read_csv("Data/information_distillation_df.csv")
final_answers_raw = pd.read_csv("Data/final_answer_df.csv")

avg_query_eff = []
avg_nav_eff = []
avg_distill_eff = []
success_category = []

for _, row in final_answers_combined.iterrows():
    session_queries = query_formulation.loc[(query_formulation["uid"]==row["uid"]) & (query_formulation["topic_number"]==row["topic_number"])]
    session_serp_navs = navigation_quality.loc[(navigation_quality["uid"]==row["uid"]) & (navigation_quality["topic_number"]==row["topic_number"])]
    session_explored_pages = information_distillation.loc[(information_distillation["uid"]==row["uid"]) & (information_distillation["topic_number"]==row["topic_number"])]

    avg_query_eff.append(session_queries["effective_query"].mean())
    avg_nav_eff.append(session_serp_navs["effective_navigation"].mean())
    avg_distill_eff.append(session_explored_pages["effective_distillation"].mean())

    if row["answer_match_ratio"] < 1:
        success_category.append("fail")
    else:
        success_category.append("success")

final_answers_combined["avg_query_eff"] = avg_query_eff
final_answers_combined["avg_nav_eff"] = avg_nav_eff
final_answers_combined["avg_distill_eff"] = avg_distill_eff
final_answers_combined["success_category"] = success_category

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

final_answers_combined = final_answers_combined.sort_values(by=["success_category", "uid", "topic_number"], ignore_index=True)

# Create row index for x-axis
df_plot = final_answers_combined.reset_index(names="row_id")

# Reshape to long format
df_long = df_plot.melt(
    id_vars=["row_id", "success_category"],
    value_vars=["avg_query_eff", "avg_nav_eff", "avg_distill_eff"],
    var_name="metric",
    value_name="effectiveness"
)

plt.figure(figsize=(14, 6))

# metrics = sorted(df_long["metric"].unique())

# offsets = {
#     metric: offset
#     for metric, offset in zip(
#         metrics,
#         np.linspace(-0.2, 0.2, len(metrics))
#     )
# }

# df_plot = df_long.copy()
# df_plot["row_id_offset"] = (
#     df_plot["row_id"]
#     + df_plot["metric"].map(offsets)
# )

sns.scatterplot(
    data=df_long,
    x="row_id",
    y="effectiveness",
    hue="success_category",   # color by success category
    style="metric",           # marker by metric
    s=100,
    alpha=0.6
)

plt.xticks(
    ticks=df_plot["row_id"],
    labels=df_plot["row_id"]+1,
    rotation=45
)

plt.xlabel("Session ID")
plt.ylabel("Effectiveness")
plt.title("Effectiveness Metrics per Session")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.preprocessing import MinMaxScaler

categories = list(final_answers_combined.drop(columns=["uid", "topic_number", "prompt", "answer", "TaskStartTime", "TaskEndTime", "matched_keywords", "num_answer_matches", "answer_match_ratio", "success_category"]).columns)
final_answers_combined = final_answers_combined.sort_values(by=["success_category", "uid", "topic_number"], ignore_index=True)
final_answers_combined[categories] = final_answers_combined[categories].apply(zscore, nan_policy="omit", ddof=1)

# Create row index for x-axis
df_plot = final_answers_combined.reset_index(names="row_id")

# Convert to long format
df_long = df_plot.melt(
    id_vars=["row_id", "success_category"],
    value_vars=categories,
    var_name="metric",
    value_name="efficiency"
)

# df_long

plt.figure(figsize=(50, 6))

sns.scatterplot(
    data=df_long,
    x="row_id",
    y="efficiency",
    hue="success_category",   # color by success category
    style="metric",           # marker by metric
    s=100,
    alpha=0.6
)

plt.xlabel("Row")
plt.ylabel("Efficiency")
plt.title("Efficiency Metrics per Row")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

plt.show()